# 🔧 ML Data Ingestion Pipeline — RiskBricks Stock Forecast
Ingests and computes all features needed for the stock direction ensemble model.
Registered model: `riskbricks.models.stock_forecast_ensemble` (LightGBM + RandomForest + GradientBoosting)

**Data Sources:**
| Source | Table | Description |
|--------|-------|-------------|
| Yahoo RSS | `riskbricks.bronze.news_rss_all` | Stock-specific news headlines |
| GDELT | `riskbricks.bronze.historical_news_gdelt` | Geopolitical events |
| FRED | `riskbricks.bronze.fred_macro_indicators` | VIX, yield spread, credit spreads |
| Stock Prices | `riskbricks.silver.stock_prices` | OHLCV for technical indicators |
| Computed | `riskbricks.silver.ml_training_features` | Final 31-feature training set |

**Schedule:** Run daily at market close (4:30 PM ET)

In [0]:
%pip install feedparser beautifulsoup4 mlflow lightgbm scikit-learn --quiet

In [0]:
# ── Import centralized config ────────────────────────────────────────
import sys, os
_nb  = dbutils.entry_point.getDbutils().notebook().getContext().notebookPath().get()
_root = "/Workspace" + _nb[:_nb.find("/notebooks/")]
sys.path.insert(0, _root)
from config import (
    CATALOG as _CFG_CATALOG, CURATED_FEATURES, FRED_SERIES,
    get_symbols, get_sector_map, get_company_names, sym_list_sql,
    setup_logger, log_step,
)

from pyspark.sql import functions as F, Window
from datetime import datetime, timedelta
import requests, csv, io, time, feedparser
import pandas as pd

dbutils.widgets.text("catalog", "riskbricks")
CATALOG = dbutils.widgets.get("catalog").strip()

# ── Dynamic symbol loading (full Fortune 500) ───────────────────────
FOCUS_SYMBOLS = get_symbols(spark, CATALOG)
SECTOR_MAP = get_sector_map(spark, CATALOG)
COMPANY_NAMES = get_company_names(spark, CATALOG)

today = datetime.now().strftime("%Y-%m-%d")
sym_list = sym_list_sql(FOCUS_SYMBOLS)
print(f"\u2705 Config loaded: {len(FOCUS_SYMBOLS)} symbols, {len(CURATED_FEATURES)} features, date={today}")

## 1. RSS News Scraping
Scrape Yahoo Finance RSS + Google News RSS for stock-specific headlines.

In [0]:
# ── 1. RSS News Scraping ────────────────────────────────────────
from bs4 import BeautifulSoup
from email.utils import parsedate_to_datetime

articles = []
for sym in FOCUS_SYMBOLS:
    company = COMPANY_NAMES.get(sym, sym)
    for feed_url in [
        f"https://feeds.finance.yahoo.com/rss/2.0/headline?s={sym}&region=US&lang=en-US",
        f"https://news.google.com/rss/search?q={company}+stock&hl=en-US&gl=US&ceid=US:en",
    ]:
        try:
            feed = feedparser.parse(feed_url)
            for entry in feed.entries:
                pub_date = None
                if hasattr(entry, "published"):
                    try: pub_date = parsedate_to_datetime(entry.published).strftime("%Y-%m-%d")
                    except Exception:  pass
                articles.append({
                    "symbol": sym, "company_name": company,
                    "title": BeautifulSoup(entry.get("title",""), "html.parser").get_text()[:500],
                    "url": entry.get("link",""), "source": feed_url.split("/")[2],
                    "published_date": pub_date,
                })
        except Exception:  pass

if articles:
    rss_df = spark.createDataFrame(pd.DataFrame(articles))
    rss_df = rss_df.withColumn("published_date", F.col("published_date").cast("date")) \
                    .withColumn("ingestion_timestamp", F.current_timestamp()) \
                    .withColumn("sector", F.lit(None).cast("string"))
    rss_df.write.mode("append").saveAsTable(f"{CATALOG}.bronze.news_rss_all")
    print(f"✅ Scraped {len(articles)} RSS articles, appended to {CATALOG}.bronze.news_rss_all")
else:
    print("⚠️ No RSS articles scraped (possible network restriction on serverless)")


## 2. FRED Macro Data
Fetch VIX, yield spread, HY credit spread, 10Y treasury from FRED (free, no API key).

In [0]:
# ── 2. FRED Macro Indicators ───────────────────────────────────
FRED_SERIES = {
    # FRED series loaded from config (see cell 1)
    "DFF": "Fed_Funds_Rate", "BAMLH0A0HYM2": "HY_Credit_Spread",
    "DGS10": "Treasury_10Y", "DGS2": "Treasury_2Y",
    "DTWEXBGS": "USD_Index", "DCOILWTICO": "WTI_Oil",
}

end_date = today
start_date = (datetime.now() - timedelta(days=30)).strftime("%Y-%m-%d")
macro_rows = []

for series_id, col_name in FRED_SERIES.items():
    try:
        url = f"https://fred.stlouisfed.org/graph/fredgraph.csv?id={series_id}&cosd={start_date}&coed={end_date}"
        resp = requests.get(url, timeout=10)
        if resp.status_code == 200:
            reader = csv.DictReader(io.StringIO(resp.text))
            for row in reader:
                date_val = list(row.values())[0]
                val_str = list(row.values())[1]
                if val_str and val_str != "." and val_str.strip():
                    try: macro_rows.append({"indicator": col_name, "date": date_val, "value": float(val_str)})
                    except Exception:  pass
            print(f"  ✅ {col_name} ({series_id})")
    except Exception as e:
        print(f"  ❌ {col_name}: {e}")

if macro_rows:
    macro_sdf = spark.createDataFrame(pd.DataFrame(macro_rows))
    macro_sdf = macro_sdf.withColumn("date", F.col("date").cast("date"))
    macro_sdf = macro_sdf.withColumn("ingestion_timestamp", F.current_timestamp())
    macro_sdf.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{CATALOG}.bronze.fred_macro_indicators")
    print(f"✅ FRED: {len(macro_rows)} observations → {CATALOG}.bronze.fred_macro_indicators")


## 3. Technical Indicators
RSI(14), MACD histogram, Bollinger Band %, volume ratio, overnight gap, close position — all from existing stock_prices.

In [0]:
# ── 3. Technical Indicators ─────────────────────────────────────
w = Window.partitionBy("symbol").orderBy("date")
w14 = Window.partitionBy("symbol").orderBy("date").rowsBetween(-13, 0)
w20 = Window.partitionBy("symbol").orderBy("date").rowsBetween(-19, 0)
w12 = Window.partitionBy("symbol").orderBy("date").rowsBetween(-11, 0)
w26 = Window.partitionBy("symbol").orderBy("date").rowsBetween(-25, 0)
w9  = Window.partitionBy("symbol").orderBy("date").rowsBetween(-8, 0)
w5  = Window.partitionBy("symbol").orderBy("date").rowsBetween(-4, 0)

prices_raw = spark.sql(f"""
    SELECT symbol, DATE(date) AS date, open, high, low, close, volume
    FROM {CATALOG}.silver.stock_prices
    WHERE symbol IN ({sym_list}) AND DATE(date) >= DATE_SUB(CURRENT_DATE(), 60)
""")

tech = (prices_raw
    .withColumn("prev_close", F.lag("close").over(w))
    .withColumn("change", F.col("close") - F.col("prev_close"))
    .withColumn("gain", F.when(F.col("change") > 0, F.col("change")).otherwise(0))
    .withColumn("loss", F.when(F.col("change") < 0, F.abs(F.col("change"))).otherwise(0))
    .withColumn("avg_gain_14", F.avg("gain").over(w14))
    .withColumn("avg_loss_14", F.avg("loss").over(w14))
    .withColumn("rsi_14", 100 - (100 / (1 + F.col("avg_gain_14") / F.greatest(F.col("avg_loss_14"), F.lit(0.0001)))))
    .withColumn("ema_12", F.avg("close").over(w12))
    .withColumn("ema_26", F.avg("close").over(w26))
    .withColumn("macd_line", F.col("ema_12") - F.col("ema_26"))
    .withColumn("macd_signal", F.avg("macd_line").over(w9))
    .withColumn("macd_hist", F.col("macd_line") - F.col("macd_signal"))
    .withColumn("bb_mid", F.avg("close").over(w20))
    .withColumn("bb_std", F.stddev("close").over(w20))
    .withColumn("bb_pct", (F.col("close") - (F.col("bb_mid") - 2*F.col("bb_std"))) / F.greatest((4*F.col("bb_std")), F.lit(0.01)))
    .withColumn("avg_vol_20", F.avg("volume").over(w20))
    .withColumn("vol_ratio", F.col("volume") / F.greatest(F.col("avg_vol_20"), F.lit(1)))
    .withColumn("daily_range", (F.col("high") - F.col("low")) / F.col("close"))
    .withColumn("avg_range_5", F.avg("daily_range").over(w5))
    .withColumn("gap_pct", (F.col("open") - F.col("prev_close")) / F.col("prev_close"))
    .withColumn("close_position", (F.col("close") - F.col("low")) / F.greatest(F.col("high") - F.col("low"), F.lit(0.01)))
)

tech_features = tech.select("symbol", "date",
    F.round("rsi_14",2).alias("rsi_14"), F.round("macd_hist",4).alias("macd_hist"),
    F.round("bb_pct",3).alias("bb_pct"), F.round("vol_ratio",2).alias("vol_ratio"),
    F.round("avg_range_5",4).alias("avg_range_5"), F.round("gap_pct",4).alias("gap_pct"),
    F.round("close_position",3).alias("close_position"))

tech_features = tech_features.withColumn("computed_at", F.current_timestamp())
tech_features.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{CATALOG}.silver.technical_indicators")
print(f"✅ Technical indicators → {CATALOG}.silver.technical_indicators ({tech_features.count()} rows)")

## 4. Sector Features + Market Breadth
Sector-relative momentum, sector breadth, advance/decline ratio from full 412-stock universe.

In [0]:
# ── 4a. Sector Features ─────────────────────────────────────
sector_df = spark.createDataFrame([(k, v) for k, v in SECTOR_MAP.items()], ["symbol", "sector"])
prices_daily = spark.sql(f"""
    SELECT symbol, DATE(date) AS date, close,
           (close - LAG(close) OVER (PARTITION BY symbol ORDER BY date)) /
           LAG(close) OVER (PARTITION BY symbol ORDER BY date) AS daily_return
    FROM {CATALOG}.silver.stock_prices
    WHERE symbol IN ({sym_list}) AND DATE(date) >= DATE_SUB(CURRENT_DATE(), 30)
""")

ps = prices_daily.join(sector_df, "symbol")
sector_avg = ps.groupBy("sector", "date").agg(
    F.avg("daily_return").alias("sector_avg_return"),
    F.count("*").alias("sector_count"),
    F.sum(F.when(F.col("daily_return") > 0, 1).otherwise(0)).alias("sector_up"))

svs = ps.join(sector_avg, ["sector", "date"]) \
    .withColumn("stock_vs_sector", F.col("daily_return") - F.col("sector_avg_return")) \
    .withColumn("sector_breadth", F.col("sector_up") / F.col("sector_count"))

w5s = Window.partitionBy("symbol").orderBy("date").rowsBetween(-4, 0)
w5sec = Window.partitionBy("sector").orderBy("date").rowsBetween(-4, 0)
svs = svs.withColumn("sector_rel_5d", F.sum("stock_vs_sector").over(w5s)) \
    .withColumn("sector_momentum_5d", F.sum("sector_avg_return").over(w5sec))

sector_out = svs.select("symbol", "date", "sector",
    F.round("sector_rel_5d",4).alias("sector_rel_5d"),
    F.round("sector_momentum_5d",4).alias("sector_momentum_5d"),
    F.round("sector_breadth",3).alias("sector_breadth"),
    F.round("stock_vs_sector",4).alias("stock_vs_sector_1d"))
sector_out = sector_out.withColumn("computed_at", F.current_timestamp())
sector_out.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{CATALOG}.silver.sector_features")
print(f"✅ Sector features → {CATALOG}.silver.sector_features")

# ── 4b. Market Breadth ──────────────────────────────────────
breadth = spark.sql(f"""
    WITH daily AS (
        SELECT DATE(date) AS date, symbol, close,
               AVG(close) OVER (PARTITION BY symbol ORDER BY date ROWS BETWEEN 19 PRECEDING AND CURRENT ROW) AS ma_20,
               (close - LAG(close) OVER (PARTITION BY symbol ORDER BY date)) / LAG(close) OVER (PARTITION BY symbol ORDER BY date) AS ret
        FROM {CATALOG}.silver.stock_prices WHERE DATE(date) >= DATE_SUB(CURRENT_DATE(), 30)
    )
    SELECT date, AVG(ret) AS market_return,
           SUM(CASE WHEN ret > 0 THEN 1 ELSE 0 END) / COUNT(*) AS advance_ratio,
           SUM(CASE WHEN close > ma_20 THEN 1 ELSE 0 END) / COUNT(*) AS pct_above_ma20,
           STDDEV(ret) AS market_dispersion
    FROM daily WHERE ret IS NOT NULL GROUP BY date
""")
breadth = breadth.withColumn("computed_at", F.current_timestamp())
breadth.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{CATALOG}.silver.market_breadth")
print(f"✅ Market breadth → {CATALOG}.silver.market_breadth")

## 5. AI Sentiment Scoring
Score RSS headlines with `ai_classify()` for stock-specific sentiment.

In [0]:
# ── 5. AI Sentiment Scoring ─────────────────────────────────
# Score articles from last 3 days for today's prediction
window_start = (datetime.now() - timedelta(days=3)).strftime("%Y-%m-%d")

scored = spark.sql(f"""
    WITH classified AS (
        SELECT symbol,
               ai_classify(title, ARRAY('very_positive','positive','neutral','negative','very_negative')) AS sent
        FROM {CATALOG}.bronze.news_rss_all
        WHERE symbol IN ({sym_list})
          AND published_date BETWEEN '{window_start}' AND '{today}'
          AND title IS NOT NULL AND LENGTH(title) > 10
    )
    SELECT symbol,
           CAST(AVG(CASE sent WHEN 'very_positive' THEN 2.0 WHEN 'positive' THEN 1.0
                WHEN 'neutral' THEN 0.0 WHEN 'negative' THEN -1.0 WHEN 'very_negative' THEN -2.0 END) AS DOUBLE) AS ai_sentiment,
           CAST(COUNT(*) AS BIGINT) AS news_count,
           CAST(SUM(CASE WHEN sent IN ('positive','very_positive') THEN 1 ELSE 0 END) AS BIGINT) AS pos_articles,
           CAST(SUM(CASE WHEN sent IN ('negative','very_negative') THEN 1 ELSE 0 END) AS BIGINT) AS neg_articles
    FROM classified GROUP BY symbol
""")
scored = scored.withColumn("as_of_date", F.current_date()).withColumn("computed_at", F.current_timestamp())
scored.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{CATALOG}.silver.news_ai_sentiment")
print(f"✅ AI sentiment scored for {scored.count()} symbols → {CATALOG}.silver.news_ai_sentiment")

## 6. Assemble ML Feature Vector
Join all sources into the final training/prediction feature table.

In [0]:
# ── 6. Assemble ML Features ─────────────────────────────────
features = spark.sql(f"""
    WITH prices AS (
        SELECT symbol, last_close, return_5d, return_20d, volatility_20d, as_of_date
        FROM {CATALOG}.silver.forecast_features_daily
        WHERE symbol IN ({sym_list})
          AND as_of_date = (SELECT MAX(as_of_date) FROM {CATALOG}.silver.forecast_features_daily)
    ),
    tech AS (
        SELECT * FROM {CATALOG}.silver.technical_indicators
        WHERE date = (SELECT MAX(date) FROM {CATALOG}.silver.technical_indicators WHERE symbol IN ({sym_list}))
    ),
    sector AS (
        SELECT * FROM {CATALOG}.silver.sector_features
        WHERE date = (SELECT MAX(date) FROM {CATALOG}.silver.sector_features)
    ),
    breadth AS (
        SELECT * FROM {CATALOG}.silver.market_breadth
        WHERE date = (SELECT MAX(date) FROM {CATALOG}.silver.market_breadth)
    ),
    macro AS (
        SELECT indicator, value FROM {CATALOG}.bronze.fred_macro_indicators
        WHERE date = (SELECT MAX(date) FROM {CATALOG}.bronze.fred_macro_indicators WHERE indicator = 'VIX')
    ),
    ai AS (SELECT * FROM {CATALOG}.silver.news_ai_sentiment),
    gdelt AS (
        SELECT symbol, AVG(avg_tone) AS gdelt_tone, COUNT(*) AS gdelt_events
        FROM {CATALOG}.bronze.historical_news_gdelt
        WHERE event_date >= DATE_SUB(CURRENT_DATE(), 5) AND symbol IN ({sym_list})
        GROUP BY symbol
    )
    SELECT p.symbol, p.as_of_date AS pred_date, p.last_close,
           p.return_5d, p.return_20d, p.volatility_20d,
           COALESCE(a.ai_sentiment, 0) AS ai_sentiment,
           COALESCE(a.news_count, 0) AS news_count,
           COALESCE(g.gdelt_tone, 0) AS gdelt_tone,
           COALESCE(g.gdelt_events, 0) AS gdelt_events,
           COALESCE(t.rsi_14, 50) AS rsi_14,
           COALESCE(t.macd_hist, 0) AS macd_hist,
           COALESCE(t.gap_pct, 0) AS gap_pct,
           COALESCE(s.sector_momentum_5d, 0) AS sector_momentum_5d,
           COALESCE(s.sector_breadth, 0.5) AS sector_breadth,
           COALESCE(b.advance_ratio, 0.5) AS advance_ratio,
           COALESCE(b.pct_above_ma20, 0.5) AS pct_above_ma20,
           COALESCE(MAX(CASE WHEN m.indicator='VIX' THEN m.value END), 20) AS vix,
           30 AS days_to_earnings,
           CASE WHEN DAYOFWEEK(p.as_of_date) = 2 THEN 1 ELSE 0 END AS is_monday
    FROM prices p
    LEFT JOIN ai a ON p.symbol = a.symbol
    LEFT JOIN gdelt g ON p.symbol = g.symbol
    LEFT JOIN tech t ON p.symbol = t.symbol
    LEFT JOIN sector s ON p.symbol = s.symbol
    CROSS JOIN breadth b
    CROSS JOIN macro m
    GROUP BY p.symbol, p.as_of_date, p.last_close, p.return_5d, p.return_20d, p.volatility_20d,
             a.ai_sentiment, a.news_count, g.gdelt_tone, g.gdelt_events,
             t.rsi_14, t.macd_hist, t.gap_pct, s.sector_momentum_5d, s.sector_breadth,
             b.advance_ratio, b.pct_above_ma20, p.as_of_date
""")
features.write.mode("overwrite").saveAsTable(f"{CATALOG}.gold.ml_prediction_features")
print(f"✅ ML features assembled: {features.count()} symbols → {CATALOG}.gold.ml_prediction_features")

## 7. Generate ML Predictions
Load registered ensemble model and predict for all symbols.

In [0]:
# ── 7. Generate ML Predictions ──────────────────────────────
# Train ensemble inline from training features (same hyperparams as registered model)
import numpy as np
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
import lightgbm as lgb

# Load training data
train_df = spark.table(f"{CATALOG}.silver.ml_training_features").toPandas()
y_train = train_df['actual_up'].values

for f in CURATED_FEATURES:
    if f not in train_df.columns:
        train_df[f] = 0

X_train = train_df[CURATED_FEATURES].fillna(0).values
print(f"Training on {len(X_train)} samples, {len(CURATED_FEATURES)} features...")

# Train 3 models with same hyperparams as registered model
lgb_model = lgb.LGBMClassifier(num_leaves=8, learning_rate=0.1, n_estimators=50, min_child_samples=3, random_state=42, verbose=-1)
lgb_model.fit(X_train, y_train)

rf_model = RandomForestClassifier(n_estimators=100, max_depth=5, min_samples_leaf=3, random_state=42, n_jobs=-1)
rf_model.fit(X_train, y_train)

gb_model = GradientBoostingClassifier(n_estimators=50, max_depth=3, learning_rate=0.1, min_samples_leaf=3, random_state=42)
gb_model.fit(X_train, y_train)
print("✅ Ensemble trained (LGB + RF + GB)")

# Predict on today's features
features_pdf = spark.table(f"{CATALOG}.gold.ml_prediction_features").toPandas()
for f in CURATED_FEATURES:
    if f not in features_pdf.columns:
        features_pdf[f] = 0
X_pred = features_pdf[CURATED_FEATURES].fillna(0).values

lgb_prob = lgb_model.predict_proba(X_pred)[:, 1]
rf_prob = rf_model.predict_proba(X_pred)[:, 1]
gb_prob = gb_model.predict_proba(X_pred)[:, 1]
ensemble_prob = (lgb_prob + rf_prob + gb_prob) / 3

features_pdf['direction'] = np.where(ensemble_prob > 0.5, 'UP', 'DOWN')
features_pdf['probability_up'] = np.round(ensemble_prob, 4)
features_pdf['confidence'] = np.round(np.abs(ensemble_prob - 0.5) * 2, 4)
features_pdf['lgb_prob'] = np.round(lgb_prob, 4)
features_pdf['rf_prob'] = np.round(rf_prob, 4)
features_pdf['gb_prob'] = np.round(gb_prob, 4)

# Write to gold
pred_sdf = spark.createDataFrame(features_pdf)
pred_sdf = pred_sdf.withColumn("computed_at", F.current_timestamp())
pred_sdf.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{CATALOG}.gold.ml_stock_predictions")

# Summary
n_up = (features_pdf['direction'] == 'UP').sum()
n_down = (features_pdf['direction'] == 'DOWN').sum()
hi_conf = (features_pdf['confidence'] > 0.4).sum()
print(f"\n✅ ML Predictions → {CATALOG}.gold.ml_stock_predictions")
print(f"   {n_up} UP / {n_down} DOWN | {hi_conf} high-confidence (>40%)")
print(f"\n📊 Top BUY signals (highest confidence UP):")
top_buy = features_pdf[features_pdf['direction']=='UP'].nlargest(10, 'confidence')
for _, r in top_buy.iterrows():
    print(f"   {r['symbol']:6} | UP   | conf={r['confidence']:.0%} | LGB={r['lgb_prob']:.0%} RF={r['rf_prob']:.0%} GB={r['gb_prob']:.0%}")
print(f"\n📊 Top SELL signals (highest confidence DOWN):")
top_sell = features_pdf[features_pdf['direction']=='DOWN'].nlargest(10, 'confidence')
for _, r in top_sell.iterrows():
    print(f"   {r['symbol']:6} | DOWN | conf={r['confidence']:.0%} | LGB={r['lgb_prob']:.0%} RF={r['rf_prob']:.0%} GB={r['gb_prob']:.0%}")